# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srilaya30/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [57]:
!test -d flyrank-ml-internship || git clone https://github.com/Srilaya30/flyrank-ml-internship.git

In [58]:
from pathlib import Path
import pandas as pd
import numpy as np

repo_root = Path("/content/flyrank-ml-internship")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [59]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["is_declining_label"].value_counts())
print(df["is_declining_label"].value_counts(normalize=True))

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I selected Random Forest because it can capture non-linear relationships between content and search-performance signals. It also provides feature importance, which makes the model easier to interpret. The model is used as decision support and will be compared with the Week-4 rule-based baseline using the same dataset and an honest client-grouped split.


In [60]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a client-grouped 80/20 train-test split. Pages from the same client should not appear in both training and testing because that could make the measured performance look better than it would be on unseen clients. The split therefore holds out approximately 20% of clients rather than randomly splitting individual rows.


In [61]:
from sklearn.model_selection import GroupShuffleSplit

target = "is_declining_label"
group_column = "client_id"

excluded_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

feature_columns = [
    col for col in df.columns
    if col not in excluded_columns
]

X = df[feature_columns].copy()
y = df[target].copy()
groups = df[group_column].copy()

X = X.select_dtypes(include=["number"]).copy()

print("Number of features:", X.shape[1])
print("Features:")
print(X.columns.tolist())

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", groups_train.nunique())
print("Testing clients:", groups_test.nunique())

overlap = set(groups_train) & set(groups_test)

print("Client overlap:", len(overlap))

assert len(overlap) == 0

print("✓ No client leakage detected.")

Number of features: 29
Features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: 0
✓ No client leakage detected.


In [62]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [63]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Missing values in training data:", X_train.isna().sum().sum())
print("Missing values in test data:", X_test.isna().sum().sum())

Missing values in training data: 0
Missing values in test data: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Random Forest was evaluated against the Week-4 rule-based baseline on the held-out client groups. The baseline measured a Precision@20 of 0.45, while the Random Forest measured 1.00, giving an observed absolute difference of 0.55. This shows stronger measured ranking performance for the Random Forest on this test split, although the result should be treated as directional evidence rather than proof of future performance.


In [64]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Model training completed.")

Model training completed.


In [65]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1][:k]
    return np.mean(np.asarray(y_true)[order])

model_p20 = precision_at_k(
    y_test.to_numpy(),
    y_prob,
    20
)

print("Model Precision@20:", round(model_p20, 4))

Model Precision@20: 1.0


In [66]:
baseline_test = df.iloc[test_idx].copy()

baseline_test["staleness_score"] = (
    baseline_test["days_since_last_update"]
    .rank(method="average", pct=True)
)

baseline_test["ctr_opportunity_score"] = (
    1
    - baseline_test["ctr"]
    .rank(method="average", pct=True)
)

baseline_test["baseline_score"] = (
    0.60 * baseline_test["staleness_score"]
    + 0.40 * baseline_test["ctr_opportunity_score"]
)

baseline_p20 = precision_at_k(
    baseline_test["is_declining_label"].to_numpy(),
    baseline_test["baseline_score"].to_numpy(),
    20
)

print("Baseline Precision@20:", round(baseline_p20, 4))

Baseline Precision@20: 0.45


In [67]:
comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Week-5 Random Forest"
    ],
    "Precision@20": [
        baseline_p20,
        model_p20
    ]
})

comparison["Difference"] = (
    comparison["Precision@20"]
    - baseline_p20
)

display(comparison)

,Method,Precision@20,Difference
0,Week-4 Baseline,0.45,0.00
1,Week-5 Random Forest,1.00,0.55


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest relies most on previous- and recent-period impression signals, followed by average position, days with impressions, and content age. On the held-out test set, the model made 768 errors out of 6,163 cases, including both false positives and false negatives. The model therefore captures useful directional signal, but it does not classify every page correctly and should be treated as decision support rather than a definitive prediction.


In [68]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
error_df = df.iloc[test_idx][
    ["content_id", "client_id", "trend_direction"]
].copy()

error_df["actual"] = y_test.to_numpy()
error_df["predicted"] = y_pred
error_df["probability_declining"] = y_prob

error_df["correct"] = (
    error_df["actual"] == error_df["predicted"]
)

errors = error_df[
    ~error_df["correct"]
].copy()

print("Test rows:", len(error_df))
print("Correct predictions:", error_df["correct"].sum())
print("Errors:", len(errors))
print(
    "Error rate:",
    round(len(errors) / len(error_df), 4)
)
print("Errors by actual class:")
print(errors["trend_direction"].value_counts())

Test rows: 6163
Correct predictions: 5395
Errors: 768
Error rate: 0.1246
Errors by actual class:
trend_direction
stable    376
down      336
up         56
Name: count, dtype: int64


In [69]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[2582  432]
 [ 336 2813]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.86      0.87      3014
           1       0.87      0.89      0.88      3149

    accuracy                           0.88      6163
   macro avg       0.88      0.87      0.88      6163
weighted avg       0.88      0.88      0.88      6163



In [70]:
importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print("Top 10 features:")
display(importance.head(10))

Top 10 features:


,0
impressions_prev_30d,0.204260
impressions_last_30d,0.169643
impressions_90d,0.079278
avg_position,0.057535
days_with_impressions,0.050561
content_age_days,0.048926
sessions_last_30d,0.028343
word_count,0.027802
char_count,0.027741
ctr,0.025489


In [71]:
print("========== ML-08 SELF-CHECK ==========")

print("Dataset rows:", len(df))
print("Feature count:", X.shape[1])
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", groups_train.nunique())
print("Testing clients:", groups_test.nunique())
print("Client overlap:", len(overlap))
print("Baseline Precision@20:", round(baseline_p20, 4))
print("Model Precision@20:", round(model_p20, 4))
print("Difference:", round(model_p20 - baseline_p20, 4))
print("Model errors:", len(errors))

assert len(df) == 30000
assert len(overlap) == 0
assert X_train.shape[0] > 0
assert X_test.shape[0] > 0
assert np.isfinite(baseline_p20)
assert np.isfinite(model_p20)
assert len(comparison) == 2

print("✓ ML-08 self-check passed.")

========== ML-08 SELF-CHECK ==========
Dataset rows: 30000
Feature count: 29
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: 0
Baseline Precision@20: 0.45
Model Precision@20: 1.0
Difference: 0.55
Model errors: 768
✓ ML-08 self-check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.